In [40]:
import networkx as nx
import osmnx as ox
import plotly.graph_objects as go
import plotly.io as pio
# pio.renderers.default = 'browser'
pio.renderers.default = "vscode"

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import numpy as np
import random

from scipy.optimize import linear_sum_assignment
import ot

In [41]:
def get_default_map_layout(title="NYC Map"):
    return dict(
        map=dict(
            style="carto-positron",
            center={"lat": 40.7831, "lon": -73.9712},
            zoom=12
        ),
        width = 800,
        height = 800,
        margin={"l": 0, "r": 0, "t": 0, "b": 0},
        title=title
    )

# Map download

In [42]:
# Download Manhattan street network
place_name = "Manhattan, New York City, New York, USA"
# download/model a street network for some city then visualize it
G = ox.graph.graph_from_place(place_name, network_type="drive")
G = ox.routing.add_edge_speeds(G)
G = ox.routing.add_edge_travel_times(G)
# fig, ax = ox.plot.plot_graph(G)

# === Plotly ===
# Extract the edge coordinates
edge_x = []
edge_y = []

for u, v, data in G.edges(data=True):
    if 'geometry' in data:
        xs, ys = data['geometry'].xy
        edge_x.extend(list(xs) + [None])  # Convert to list, then add None
        edge_y.extend(list(ys) + [None])
    else:
        x0, y0 = G.nodes[u]['x'], G.nodes[u]['y']
        x1, y1 = G.nodes[v]['x'], G.nodes[v]['y']
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

# Create Plotly map
fig = go.Figure(go.Scattermap(
    mode = "lines",
    lon = edge_x,
    lat = edge_y,
    line = dict(width = 1, color = 'blue'),
    hoverinfo = "none"
))

fig.update_layout(**get_default_map_layout(title="Manhattan Street Network"))

fig.show()

Reduces the graph to a strongly connected component

In [43]:
# If G is not strongly connected, extract the largest strongly connected component
if not nx.is_strongly_connected(G):
    # nx.strongly_connected_components(G) returns a generator of sets of nodes,
    # each set representing a strongly connected component.
    largest_scc = max(nx.strongly_connected_components(G), key=len)
    # Create a new graph from the largest strongly connected component
    G_scc = G.subgraph(largest_scc).copy()
else:
    # Otherwise, use G as is (or create a copy)
    G_scc = G.copy()

print("Original graph nodes:", G.number_of_nodes())
print("Nodes in largest strongly connected component:", G_scc.number_of_nodes())

Original graph nodes: 4608
Nodes in largest strongly connected component: 4520


# Data preprocessing

## 1. Filter dataset:
We select rides occurred on the 9th April 2024 that started and ended inside the Manhattan borough.

In [44]:
# === Load data ===
rides_path = "../data/processed/yellow_tripdata_2024-04.csv"
lookup_path = "../data/processed/taxi_zone_lookup.csv"

# Load the datasets
df = pd.read_csv(rides_path, low_memory=False)
zone_lookup = pd.read_csv(lookup_path)

# === Identify Manhattan zones ===
manhattan_ids = set(
    zone_lookup[zone_lookup['Borough'] == 'Manhattan']['LocationID']
)

# === Define the date to filter ===
target_date = pd.to_datetime("2024-04-09").date()


# === Filter for Manhattan-only rides ===
filtered_df = df[
    df['PULocationID'].isin(manhattan_ids) &
    df['DOLocationID'].isin(manhattan_ids) &
    (pd.to_datetime(df['tpep_pickup_datetime']).dt.date == target_date) &
    (pd.to_datetime(df['tpep_dropoff_datetime']).dt.date == target_date)
].copy()

# === Save or display result ===
filtered_df.to_csv("manhattan_only_rides.csv", index=False)
print(f"Filtered {len(filtered_df)} rides that start and end in Manhattan.")


Filtered 100288 rides that start and end in Manhattan.


## 2. Nodes-zone matching:

For each zone we identify the set of nodes that belongs to it. The obtained sets are interactively visualized in the map.

In [45]:
# Get graph nodes as GeoDataFrame
nodes = ox.graph_to_gdfs(G_scc, nodes=True, edges=False)
nodes['geometry'] = nodes.apply(lambda row: Point(row['x'], row['y']), axis=1)
nodes_gdf = gpd.GeoDataFrame(nodes, geometry='geometry', crs="EPSG:4326")

gdf_zones = gpd.read_file("../data/processed/taxi_zones/taxi_zones.shp")
gdf_zones = gdf_zones.to_crs("EPSG:4326")   # ensure same CRS as nodes

print(gdf_zones.columns)

# Spatial join: which nodes fall into which zone
nodes_in_zones = gpd.sjoin(nodes_gdf, gdf_zones, how="inner", predicate="within")

# DataFrame with node IDs and their corresponding zone
zone_to_nodes = {
    zone: list(group.index)
    for zone, group in nodes_in_zones.groupby('zone')
}


Index(['OBJECTID', 'Shape_Leng', 'Shape_Area', 'zone', 'LocationID', 'borough',
       'geometry'],
      dtype='object')


In [46]:
# This assumes zone_to_nodes maps zone names or IDs → list of node IDs
node_to_zone = {
    node: zone for zone, nodes in zone_to_nodes.items() for node in nodes
}
nodes_gdf['zone'] = nodes_gdf.index.map(node_to_zone)
colored_nodes = nodes_gdf.dropna(subset=['zone'])

import plotly.graph_objects as go
import pandas as pd

fig = go.Figure()

# One scattermapbox layer per zone
for zone_name, group in colored_nodes.groupby('zone'):
    fig.add_trace(go.Scattermap(
        mode="markers",
        lat=group.geometry.y,
        lon=group.geometry.x,
        marker=dict(size=6),
        name=str(zone_name),
        hovertext=zone_name
    ))

# Update layout
fig.update_layout(**get_default_map_layout(title="Manhattan Street Network with Zones"))

fig.show()


## 3. BPR congestion model

We use a simplified BPR-inspired multiplier, `t_congested = t_freeflow × α`:

```python 
bpr_congestion_multipliers = {
    'motorway': 1.2,
    'trunk': 1.3,
    'primary': 1.5,
    'secondary': 1.7,
    'tertiary': 1.9,
    'residential': 2.0,
    'unclassified': 2.0,
    'service': 2.2,
    'living_street': 2.5,
    'default': 2.0
}

In [47]:
bpr_congestion_multipliers = {
    'motorway': 1.2,
    'trunk': 1.3,
    'primary': 1.5,
    'secondary': 1.7,
    'tertiary': 1.9,
    'residential': 2.0,
    'unclassified': 2.0,
    'service': 2.2,
    'living_street': 2.5,
    'default': 2.0
}

# === Compute t_freeflow and apply congestion to each edge ===
for u, v, k, data in G_scc.edges(keys=True, data=True):
    speed_kph = data.get('speed_kph', 30)
    length_m = data.get('length', 0)
    
    # Free-flow time in minutes
    t_freeflow_min = (length_m / 1000) / speed_kph * 60
    
    # Get road type (may be a list)
    highway = data.get('highway', 'default')
    if isinstance(highway, list):
        highway = highway[0]

    alpha = bpr_congestion_multipliers.get(highway, bpr_congestion_multipliers['default'])
    t_congested_min = t_freeflow_min * alpha

    # Save to edge
    data['travel_time_freeflow'] = t_freeflow_min
    data['travel_time_congested'] = t_congested_min
    data['bpr_alpha'] = alpha


## 4. Demand estimation

We estimate hourly taxi demand per zone from trip data. To do so, we’ll count the number of pickups for every `(hour, zone)` combination. 

We then normalize to obtain `P(zone|hour)`

In [48]:
filtered_df['pickup_dt'] = pd.to_datetime(filtered_df['tpep_pickup_datetime'])
filtered_df['pickup_hour'] = filtered_df['pickup_dt'].dt.hour

locationID_to_nodes = {
    loc_id: list(group.index)
    for loc_id, group in nodes_in_zones.groupby('LocationID')
}

# Count trips per hour per zone
demand = filtered_df.groupby(['pickup_hour', 'PULocationID']).size().unstack(fill_value=0)

# Normalize rows to get zone probabilities for each hour
demand_prob = demand.div(demand.sum(axis=1), axis=0)

def sample_ride(hour, demand_prob, zone_to_nodes):
    if hour not in demand_prob.index:
        raise ValueError(f"No demand data for hour {hour}")
    
    zones = demand_prob.columns
    probs = demand_prob.loc[hour].values

    # Sample a zone based on empirical probabilities
    sampled_zone = np.random.choice(zones, p=probs)

    # Uniformly pick a node inside the zone
    possible_nodes = zone_to_nodes.get(sampled_zone, [])

    if not possible_nodes:
        return None, None  # skip if zone has no mapped nodes

    sampled_node = np.random.choice(possible_nodes)
    return sampled_zone, sampled_node

### Test for pick-up location sampling + visualization

In [49]:
# === Sample multiple rides upfront ===
sample_hour = 12
num_samples = 10
sampled_points = []

for _ in range(num_samples):
    zone, node = sample_ride(sample_hour, demand_prob, locationID_to_nodes)
    if node is not None:
        pt = nodes_gdf.loc[node].geometry
        sampled_points.append((pt.y, pt.x, zone))

# === Create base map with street network
fig = go.Figure()

# Street network (gray nodes)
fig.add_trace(go.Scattermap(
    mode="markers",
    lat=nodes_gdf.geometry.y,
    lon=nodes_gdf.geometry.x,
    marker=dict(size=4, color="gray"),
    name="Street Network",
    hoverinfo="skip"
))

# Sampled rides (red dots)
for i, (lat, lon, zone) in enumerate(sampled_points):
    fig.add_trace(go.Scattermap(
        mode="markers",
        lat=[lat],
        lon=[lon],
        marker=dict(size=10, color="red"),
        name=f"Ride {i+1}",
        hovertext=[f"Ride {i+1}<br>Zone {zone}"]
    ))

# === Map layout
fig.update_layout(
    map=dict(
        style="carto-positron",  # or "open-street-map"
        center={"lat": 40.7831, "lon": -73.9712},
        zoom=12,
        domain=dict(x=[0, 1], y=[0, 1])  # Full map area
    ),
    autosize=True,
    height=800,
    width=800,
    margin={"l": 0, "r": 0, "t": 0, "b": 0},
    title="Manhattan Network with Sampled Ride Pickups",
    showlegend=True
)

fig.show()

Animated `plotly` map of hourly demand.

In [50]:
# Merge total demand info into zone polygons
# If 'LocationID' is already the index, this does nothing
if 'LocationID' in gdf_zones.columns:
    gdf_zones = gdf_zones.set_index('LocationID')

# Now safely filter to zones that appear in demand.columns
gdf_zones = gdf_zones[gdf_zones.index.isin(demand.columns)]



# Compute centroids
# Reproject to projected CRS
gdf_projected = gdf_zones.to_crs("EPSG:2263")  # NY State Plane (ft)
# Compute centroids in projected CRS
gdf_projected['centroid'] = gdf_projected.geometry.centroid
# Reproject centroids back to WGS84 (EPSG:4326) for plotting
centroids = gdf_projected['centroid'].to_crs("EPSG:4326")
gdf_zones['lat'] = centroids.y
gdf_zones['lon'] = centroids.x

# Prepare a figure
fig = go.Figure()

# Create one frame per hour (0 to 23)
for hour in range(24):
    hourly_data = demand.loc[hour]
    merged = gdf_zones.copy()
    merged['demand'] = hourly_data

    fig.add_trace(go.Scattermap(
        lat=merged['lat'],
        lon=merged['lon'],
        mode='markers',
        marker=dict(
            size=merged['demand'] / merged['demand'].max() * 30 + 5,
            color=merged['demand'],
            colorscale='Reds',
            cmin=0,
            cmax=demand.values.max(),
            showscale=False
        ),
        hovertext=merged['zone'] + '<br>Demand: ' + merged['demand'].astype(int).astype(str),
        name=f"Hour {hour}",
        visible=(hour == 0)
    ))

# Create a slider to toggle visibility
steps = []
for i in range(24):
    step = dict(
        method="update",
        args=[{"visible": [j == i for j in range(24)]}],
        label=f"{i}:00"
    )
    steps.append(step)

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Hour: "},
    pad={"t": 50},
    steps=steps
)]

# Map layout
fig.update_layout(
    **get_default_map_layout(title="Manhattan Hourly Taxi Pickup Demand"),
    sliders=sliders,
)

fig.show()


# Simulator

### Initial example with only taxis moving by choosing random locations

In [51]:
# === Get next edge candidates ===
def get_random_next_edges(taxis, G_scc):
    next_edges = {}
    for i, taxi in enumerate(taxis):
        u, v, k = taxi['edge']
        successors = list(G.successors(v))  # outgoing edges from current target node
        if successors:
            # Choose one outgoing edge (v → w)
            w = np.random.choice(successors)
            # Take the first edge between v and w
            edge_data = G_scc.get_edge_data(v, w)
            if edge_data:
                edge_keys = list(edge_data.keys())
                next_edges[i] = (v, w, edge_keys[0])
    return next_edges

# === Get position of taxi along the edge ===
def interpolate_position(u, v, G_scc, progress=0.5):
    x0, y0 = G_scc.nodes[u]['x'], G_scc.nodes[u]['y']
    x1, y1 = G_scc.nodes[v]['x'], G_scc.nodes[v]['y']
    x = x0 + (x1 - x0) * progress
    y = y0 + (y1 - y0) * progress
    return y, x  # lat, lon

# === Taxi state update function ===
def update_taxi_states(taxis, dt, edge_travel_times, next_edges):
    """
    Update the state of all taxis.

    Parameters:
    - taxis: list of dicts with keys 'edge' and 'time_left'
    - dt: timestep in seconds (e.g., 15)
    - edge_travel_times: dict mapping edge (u, v, k) -> total time in seconds
    - next_edges: dict mapping taxi index -> next edge (u, v, k)

    Returns:
    - updated list of taxi states
    """
    updated_taxis = []

    for taxi in taxis:
        remaining_dt = dt

        # Continue updating this taxi until the full dt is consumed
        while remaining_dt > 0:

            # If the taxi can finish its current edge within the remaining dt:
            if taxi['time_left'] <= remaining_dt:
                # Subtract the time to finish the current edge
                remaining_dt -= taxi['time_left']
                # The taxi finishes this edge; choose a next edge.
                u, v, k = taxi['edge']
                # Get outgoing neighbors (edges) from the node v
                neighbors = list(G_scc.out_edges(v, keys=True))

                if neighbors:
                    # Choose a next edge at random
                    new_edge = random.choice(neighbors)
                else:
                    # If there are no available next edges, remain on current edge.
                    new_edge = taxi['edge']
                taxi['edge'] = new_edge
                taxi['time_left'] = edge_travel_times[new_edge]

            else:
                # The taxi is still mid-edge; simply subtract the remaining dt.
                taxi['time_left'] -= remaining_dt
                remaining_dt = 0
                
        updated_taxis.append(taxi)
    return updated_taxis


In [52]:
# === PARAMETERS ===
num_taxis = 100
dt = 10  # seconds
num_steps = 100

# === Prepare travel time dictionary ===
edge_travel_times = {
    (u, v, k): data['travel_time_congested'] * 60
    for u, v, k, data in G_scc.edges(keys=True, data=True)
}

# === Sample initial taxi states ===
valid_edges = list(edge_travel_times.keys())
initial_edges = random.sample(valid_edges, k=num_taxis)

taxis = [{'edge': edge, 'time_left': edge_travel_times[edge]} for edge in initial_edges]

# === Simulate + store positions for each timestep ===
taxi_positions_per_step = []

for step in range(num_steps):
    # Compute positions
    positions = []
    for taxi in taxis:
        u, v, k = taxi['edge']
        total_time = edge_travel_times[(u, v, k)]
        time_left = taxi['time_left']
        progress = 1 - (time_left / total_time) if total_time > 0 else 0
        lat, lon = interpolate_position(u, v, G_scc, progress)
        positions.append((lat, lon))

    taxi_positions_per_step.append(positions)

    # Update taxis
    next_edges = get_random_next_edges(taxis, G_scc)
    taxis = update_taxi_states(taxis, dt, edge_travel_times, next_edges)

# === Create Plotly frames for animation ===
frames = []
for t, positions in enumerate(taxi_positions_per_step):
    lat, lon = zip(*positions)
    frame = go.Frame(
        data=[go.Scattermap(
            lat=lat,
            lon=lon,
            mode='markers',
            marker=dict(size=10, color='red'),
            name="Taxis"
        )],
        name=str(t)
    )
    frames.append(frame)

# === Base figure with background nodes ===
fig = go.Figure()

# === Add first taxi positions ===
lat0, lon0 = zip(*taxi_positions_per_step[0])
fig.add_trace(go.Scattermap(
    lat=nodes_gdf.geometry.y,
    lon=nodes_gdf.geometry.x,
    mode='markers',
    marker=dict(size=4),
    name="Network",
    hoverinfo="skip"
))

# === Layout with animation ===
fig.update_layout(
    **get_default_map_layout(title="Taxi Fleet Simulation"),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        buttons=[dict(label="Play", method="animate", args=[None])]
    )],
    sliders=[dict(
        steps=[dict(method="animate", args=[[str(i)]], label=f"{i}")
               for i in range(num_steps)],
        transition=dict(duration=0),
        x=0.1, xanchor="left", y=0, yanchor="top"
    )]
)
fig.frames = frames  # Set frames separately here


fig.show()

In [53]:
print('travel time:', sorted(edge_travel_times.values()))

travel time: [np.float64(0.45716285687819025), np.float64(0.45914507284510087), np.float64(0.5096921669286195), np.float64(0.5769552393025384), np.float64(0.59552818783484), np.float64(0.59552818783484), np.float64(0.6563978534203625), np.float64(0.6785298550643533), np.float64(0.8410218307135142), np.float64(0.8578311169316912), np.float64(0.8606961818345356), np.float64(0.8657299228178568), np.float64(0.9051457839570871), np.float64(0.9126943742250725), np.float64(0.9126943742250725), np.float64(0.912759160133809), np.float64(0.9151476635344213), np.float64(0.9328868555980205), np.float64(0.9540472811745038), np.float64(0.9668854938810834), np.float64(0.9780192148790225), np.float64(0.9804031459140726), np.float64(1.0123819158953828), np.float64(1.0355949748311875), np.float64(1.0355949748311875), np.float64(1.0593963807881734), np.float64(1.081867282185867), np.float64(1.081867282185867), np.float64(1.0879442485960267), np.float64(1.0953010661270395), np.float64(1.0953010661270395),

### Adding ride requests and matching

1. Assuming fixed demand distribution (e.g. 17:00), sample a number of ride requests equal to your number of taxis. 

In [54]:
hour = 17
ride_requests = []
ride_nodes = []  # pickup node for matching

def sample_ride(hour, demand_prob, zone_to_nodes):
    """
    Samples a ride by first choosing a zone based on the demand distribution,
    then uniformly selecting a node from that zone.
    """
    if hour not in demand_prob.index:
        raise ValueError(f"No demand data for hour {hour}")
    
    zones = demand_prob.columns
    probs = demand_prob.loc[hour].values

    sampled_zone = np.random.choice(zones, p=probs)
    # Uniformly pick a node within the sampled zone
    possible_nodes = zone_to_nodes.get(sampled_zone, [])
    if not possible_nodes:
        return None, None
    sampled_node = np.random.choice(possible_nodes)
    return sampled_zone, sampled_node


def sample_reachable_ride(hour, demand_prob, zone_to_nodes, driver_nodes, G_scc, max_attempts=10):
    """
    Samples a ride that is reachable by at least one driver.
    
    Parameters:
      hour: Hour of demand (must exist in demand_prob index)
      demand_prob: DataFrame with normalized demand probabilities per zone.
      zone_to_nodes: Dictionary mapping zone IDs (e.g. PULocationID) to lists of node IDs from G.
      driver_nodes: List of driver nodes (IDs in G).
      G: The OSMnx graph.
      max_attempts: Maximum number of sampling attempts.
      
    Returns:
      (sampled_zone, sampled_node) if a reachable ride is found, else (None, None)
    """
    for attempt in range(max_attempts):
        # Sample a ride using your existing sample_ride function
        sampled_zone, sampled_node = sample_ride(hour, demand_prob, zone_to_nodes)
        if sampled_node is None:
            continue  # try another sample if no node was returned
        
        # Check if at least one driver can reach the sampled node.
        reachable = False
        for d in driver_nodes:
            try:
                if nx.has_path(G_scc, d, sampled_node):
                    reachable = True
                    break
            except Exception as e:
                # if an error occurs, treat the ride as unreachable
                continue
                
        if reachable:
            return sampled_zone, sampled_node
        else:
            print(f"Sampled node {sampled_node} in zone {sampled_zone} is unreachable; resampling...")
    # If no ride is reached after max_attempts, return None values.
    return None, None

def sample_ride_for_driver(driver_node, hour, demand_prob, zone_to_nodes, G_scc, max_attempts=10):
    """
    Samples a ride that is reachable by the specified driver.
    """
    for attempt in range(max_attempts):
        sampled_zone, sampled_node = sample_ride(hour, demand_prob, zone_to_nodes)
        if sampled_node is None:
            continue

        try:
            if nx.has_path(G_scc, driver_node, sampled_node):
                return sampled_zone, sampled_node
        except Exception:
            continue
    return None, None

2. Determine driver positions and compute shortest path costs

Simulation with shortest path and matching

In [55]:
# === PARAMETERS ===
num_taxis = 20    # Number of taxis (and pickups) to simulate.
dt = 10           # Timestep duration in seconds.
num_steps = 100   # Number of simulation timesteps.

# === ASSUMPTIONS ===
# - G_scc is a pre-defined NetworkX graph (e.g., a MultiDiGraph).
# - Each node in G_scc has attributes 'x' (longitude) and 'y' (latitude).
# - Each edge in G_scc has:
#      • 'travel_time_congested': travel time in minutes under congestion
#      • (optionally) other attributes (e.g., 'length')
#
# For this example, we assume G_scc exists. If not, load or create it first:
# G_scc = nx.read_gpickle("your_graph_file.gpickle")

# === HELPER FUNCTIONS ===

def compute_shortest_route(G, source, target):
    """
    Compute the shortest route (as a list of edge dictionaries) from source to target
    using the 'travel_time_congested' attribute as weight. Each edge dict contains:
       - 'start': source node,
       - 'end': target node,
       - 'key': edge key (for multigraphs),
       - 'congested_time': travel time on the edge in seconds.
    """
    try:
        path = nx.shortest_path(G, source=source, target=target, weight='travel_time_congested')
    except nx.NetworkXNoPath:
        return []
    
    route = []
    for i in range(len(path) - 1):
        u = path[i]
        v = path[i+1]
        # For multigraphs, select the first available edge.
        k = list(G[u][v].keys())[0]
        edge_data = G[u][v][k]
        congested_time = edge_data['travel_time_congested'] * 60  # Convert minutes to seconds.
        route.append({
            'start': u,
            'end': v,
            'key': k,
            'congested_time': congested_time
        })
    return route

def interpolate_position_on_edge(start, end, G, progress):
    """
    Interpolate linearly between nodes `start` and `end` given a progress fraction (0 to 1)
    along the edge. Returns (lat, lon).
    """
    lat1, lon1 = G.nodes[start]['y'], G.nodes[start]['x']
    lat2, lon2 = G.nodes[end]['y'], G.nodes[end]['x']
    lat = lat1 + (lat2 - lat1) * progress
    lon = lon1 + (lon2 - lon1) * progress
    return lat, lon

def update_taxi_states_dt(dt, taxis):
    """
    For each taxi, update its state using the timestep dt (in seconds) as follows:
      - Each taxi has a current edge with a known remaining congested time.
      - While dt_remaining is positive, subtract the congested time from dt_remaining 
        for each edge traveled.
      - When dt_remaining is not sufficient to finish the current edge, interpolate 
        the taxi's position along that edge.
      
    Returns the updated taxi list.
    """
    for taxi in taxis:
        if taxi['state'] == 'moving':
            dt_remaining = dt
            # If the taxi is not on an edge but has a planned route, assign the next edge.
            if taxi['current_edge'] is None and taxi['route']:
                taxi['current_edge'] = taxi['route'].pop(0)
                taxi['remaining_time_on_edge'] = taxi['current_edge']['congested_time']
            # Consume dt along the route.
            while dt_remaining > 0 and taxi['current_edge'] is not None:
                if dt_remaining >= taxi['remaining_time_on_edge']:
                    dt_remaining -= taxi['remaining_time_on_edge']
                    taxi['current_location'] = (
                        G_scc.nodes[taxi['current_edge']['end']]['y'],
                        G_scc.nodes[taxi['current_edge']['end']]['x']
                    )
                    if taxi['route']:
                        taxi['current_edge'] = taxi['route'].pop(0)
                        taxi['remaining_time_on_edge'] = taxi['current_edge']['congested_time']
                    else:
                        taxi['current_edge'] = None
                        taxi['remaining_time_on_edge'] = 0
                        taxi['state'] = 'idle'
                        break
                else:
                    taxi['remaining_time_on_edge'] -= dt_remaining
                    full_time = taxi['current_edge']['congested_time']
                    progress = (full_time - taxi['remaining_time_on_edge']) / full_time
                    taxi['current_location'] = interpolate_position_on_edge(
                        taxi['current_edge']['start'],
                        taxi['current_edge']['end'],
                        G_scc,
                        progress
                    )
                    dt_remaining = 0
    return taxis


In [56]:

# === STEP 1: SAMPLE TAXI POSITIONS AND PICKUP LOCATIONS SEPARATELY ===

all_nodes = list(G_scc.nodes())
# Sample taxi starting positions (nodes) and pickup locations (nodes) separately.
taxi_nodes = random.sample(all_nodes, num_taxis)
pickup_nodes = random.sample(all_nodes, num_taxis)

# === STEP 2: BUILD COST MATRIX FOR OPTIMAL MATCHING ===

# The cost between taxi i and pickup j is defined as the shortest-path cost (using
# 'travel_time_congested' as the weight) from taxi_nodes[i] to pickup_nodes[j].
cost_matrix = np.zeros((num_taxis, num_taxis))
for i, t_node in enumerate(taxi_nodes):
    for j, p_node in enumerate(pickup_nodes):
        try:
            # nx.shortest_path_length returns the sum of edge weights.
            cost = nx.shortest_path_length(G_scc, t_node, p_node, weight="travel_time_congested")
            cost_matrix[i, j] = cost * 60  # Convert minutes to seconds.
        except nx.NetworkXNoPath:
            cost_matrix[i, j] = 1e6  # Set a high cost if no path exists.

# Solve the assignment problem (Optimal Transport) using the Hungarian algorithm.
row_ind, col_ind = linear_sum_assignment(cost_matrix)
# row_ind is usually [0, 1, 2, ...] and col_ind gives the matching pickup for each taxi.

# === STEP 3: BUILD THE TAXI SIMULATION BASED ON THE OPTIMAL ASSIGNMENT ===

# Build a list of taxi states with the optimal matching.
taxis = []
for idx in range(num_taxis):
    taxi_node = taxi_nodes[row_ind[idx]]
    pickup_node = pickup_nodes[col_ind[idx]]
    route = compute_shortest_route(G_scc, taxi_node, pickup_node)
    if not route:
        continue  # Skip this taxi if no route exists.
    taxi_state = {
        'id': idx,
        'route': route,                  # List of edges (each with a congested travel time in seconds)
        'current_edge': None,            # Will be set on first update.
        'remaining_time_on_edge': 0,
        'target_destination': pickup_node,
        'pickup_location': (G_scc.nodes[pickup_node]['y'], G_scc.nodes[pickup_node]['x']),
        'current_location': (G_scc.nodes[taxi_node]['y'], G_scc.nodes[taxi_node]['x']),
        'state': 'moving'
    }
    taxis.append(taxi_state)

# For later visualization, record taxi positions over time and keep track of each taxi's pickup.
taxi_positions = { taxi['id']: [] for taxi in taxis }
pickup_locations = { taxi['id']: taxi['pickup_location'] for taxi in taxis }

# === STEP 4: SIMULATION LOOP ===
for step in range(num_steps):
    # Record current positions for each taxi.
    for taxi in taxis:
        taxi_positions[taxi['id']].append(taxi['current_location'])
    # Update taxi states for the next timestep.
    taxis = update_taxi_states_dt(dt, taxis)

# === STEP 5: BUILD THE INTERACTIVE PLOTLY MAP WITH ANIMATION ===
legend_dummy = go.Scattermap(
    lat=[],
    lon=[],
    mode='markers',
    marker=dict(size=10, color='red'),
    name="Taxi",
    hoverinfo='skip',
    showlegend=True
)

pickup_traces = []
taxi_base_traces = []
first = True
for taxi_id in sorted(taxi_positions.keys()):
    # Static pickup marker (star)
    pickup_lat, pickup_lon = pickup_locations[taxi_id]
    pickup_traces.append(go.Scattermap(
        lat=[pickup_lat],
        lon=[pickup_lon],
        mode='markers',
        marker=dict(size=12, color='blue', symbol='star'),
        name=f"Pickup {taxi_id}",
        hoverinfo='text',
        hovertext=[f"Pickup {taxi_id}"],
        showlegend=True
    ))

    # Initial taxi position (red circle)
    taxi_lat, taxi_lon = taxi_positions[taxi_id][0]
    taxi_base_traces.append(go.Scattermap(
        lat=[taxi_lat],
        lon=[taxi_lon],
        mode='markers',
        marker=dict(size=10, color='red'),
        name=f"Taxi {taxi_id}",
        hoverinfo='text',
        hovertext=[f"Taxi {taxi_id}"],
        showlegend=True
    ))
first = False
# Combine both for initial frame
base_data = [legend_dummy] + taxi_base_traces + pickup_traces

# === Build animation frames for taxis only ===

frames = []
for step in range(num_steps):
    frame_data = []
    for taxi_id in sorted(taxi_positions.keys()):
        lat, lon = taxi_positions[taxi_id][step]
        frame_data.append(go.Scattermap(
            lat=[lat],
            lon=[lon],
            mode='markers',
            marker=dict(size=10, color='red'),
            name=f"Taxi {taxi_id}",
            hoverinfo='text',
            hovertext=[f"Taxi {taxi_id}"],
            showlegend=False  # legend shown only in base frame
        ))
    frames.append(go.Frame(data=frame_data, name=str(step)))

# === Layout with map and animation controls ===

fig = go.Figure(
    data=base_data,
    layout=go.Layout(
        map=dict(
            style="carto-positron",
            center={"lat": 40.7831, "lon": -73.9712},
            zoom=11,
            domain=dict(x=[0, 1], y=[0, 1])
        ),
        title="Optimal Taxi-to-Pickup Assignment Simulation",
        height=800,
        width=800,
        updatemenus=[{
            'type': 'buttons',
            'showactive': False,
            'buttons': [{
                'label': 'Play',
                'method': 'animate',
                'args': [None, {
                    'frame': {'duration': 300, 'redraw': True},
                    'fromcurrent': True,
                    'mode': 'immediate'
                }]
            }]
        }],
        sliders=[{
            'steps': [{
                'method': 'animate',
                'args': [[str(i)], {
                    'mode': 'immediate',
                    'frame': {'duration': 300, 'redraw': True},
                    'transition': {'duration': 0}
                }],
                'label': str(i)
            } for i in range(num_steps)],
            'transition': {'duration': 0},
            'x': 0.1,
            'xanchor': 'left',
            'y': 0,
            'yanchor': 'top'
        }],
        margin=dict(l=0, r=0, t=40, b=0)
    ),
    frames=frames
)

fig.show()
